# Build a small RNN to generate smiles codes

Try on two different datasets: a small toy dataset and the smiles column from the ESOL dataset (random sample of 300 points - training time estimated to be ~2-5 min - decrease the sample number if you had issues with runtime before).

In [39]:
import pandas as pd

smiles_list = [
    "CCO",
    "CCN",
    "C=O",
    "C1CCCCC1",
    "CC(=O)O",
    "CCC",
    "COC",
    "CCCl",
    "O"
]

df = pd.read_csv("esol_modified.csv").dropna(subset=["SMILES"])#.sample(300)
smiles_list2 = df["SMILES"].to_list()

Build vocabulary and tokeniser

In [40]:
chars = sorted(list(set("".join(smiles_list2))))
stoi = {ch: i for i, ch in enumerate(chars)} #string to index
itos = {i: ch for ch, i in stoi.items()} # index to string

vocab_size = len(chars)

In [41]:
print("stoi: ", stoi)
print("itos:", itos)

stoi:  {' ': 0, '#': 1, '(': 2, ')': 3, '/': 4, '1': 5, '2': 6, '3': 7, '4': 8, '5': 9, '6': 10, '7': 11, '8': 12, '=': 13, 'B': 14, 'C': 15, 'F': 16, 'H': 17, 'I': 18, 'N': 19, 'O': 20, 'P': 21, 'S': 22, '[': 23, '\\': 24, ']': 25, 'c': 26, 'l': 27, 'n': 28, 'o': 29, 'r': 30, 's': 31}
itos: {0: ' ', 1: '#', 2: '(', 3: ')', 4: '/', 5: '1', 6: '2', 7: '3', 8: '4', 9: '5', 10: '6', 11: '7', 12: '8', 13: '=', 14: 'B', 15: 'C', 16: 'F', 17: 'H', 18: 'I', 19: 'N', 20: 'O', 21: 'P', 22: 'S', 23: '[', 24: '\\', 25: ']', 26: 'c', 27: 'l', 28: 'n', 29: 'o', 30: 'r', 31: 's'}


In [48]:
def encode(smile):
    return [stoi[c] for c in smile]

data = [encode(s) for s in smiles_list2]

In [49]:
data

[[20,
  15,
  15,
  7,
  20,
  15,
  2,
  20,
  15,
  15,
  6,
  20,
  15,
  2,
  20,
  15,
  2,
  15,
  1,
  19,
  3,
  26,
  5,
  26,
  26,
  26,
  26,
  26,
  5,
  3,
  15,
  2,
  20,
  3,
  15,
  2,
  20,
  3,
  15,
  6,
  20,
  3,
  15,
  2,
  20,
  3,
  15,
  2,
  20,
  3,
  15,
  7,
  20,
  0],
 [15,
  26,
  5,
  29,
  26,
  26,
  26,
  5,
  15,
  2,
  13,
  20,
  3,
  19,
  26,
  6,
  26,
  26,
  26,
  26,
  26,
  6],
 [15, 15, 2, 15, 3, 13, 15, 15, 15, 15, 2, 15, 3, 13, 15, 15, 2, 13, 20, 3],
 [26,
  5,
  26,
  26,
  26,
  6,
  26,
  2,
  26,
  5,
  3,
  26,
  26,
  26,
  7,
  26,
  6,
  26,
  26,
  26,
  8,
  26,
  9,
  26,
  26,
  26,
  26,
  26,
  9,
  26,
  26,
  26,
  8,
  7],
 [26, 5, 26, 26, 31, 26, 5],
 [26, 6, 26, 26, 26, 5, 31, 26, 28, 26, 5, 26, 6, 0],
 [15,
  27,
  26,
  5,
  26,
  26,
  2,
  15,
  27,
  3,
  26,
  2,
  26,
  2,
  15,
  27,
  3,
  26,
  5,
  3,
  26,
  6,
  26,
  2,
  15,
  27,
  3,
  26,
  26,
  26,
  26,
  6,
  15,
  27],
 [15,
  15,
  5,
  6,
  

In [50]:
data = [seq for seq in data if len(seq) > 1]

Build Model

In [51]:
import torch
import torch.nn as nn

class SmilesRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x) # two values are returned by RNN: output and hidden state (not used)
        out = self.fc(out)
        return out

Train the RNN

In [52]:
model = SmilesRNN(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    total_loss = 0

    for seq in data:
        x = torch.tensor(seq[:-1], dtype=torch.long).unsqueeze(0)
        y = torch.tensor(seq[1:]).unsqueeze(0)

        output = model(x)
        loss = criterion(output.view(-1, vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.3f}")

Epoch 0, Loss: 1710.230
Epoch 20, Loss: 1637.708
Epoch 40, Loss: 1609.149
Epoch 60, Loss: 1629.842
Epoch 80, Loss: 1660.182
Epoch 100, Loss: 1644.459
Epoch 120, Loss: 1657.961
Epoch 140, Loss: 1702.105
Epoch 160, Loss: 1705.951
Epoch 180, Loss: 1690.699


In [53]:
import random

def generate(model, start_char='C', max_len=8):
    model.eval()
    
    idx = torch.tensor([[stoi[start_char]]])
    result = start_char

    for i in range(max_len):
        output = model(idx)
        probs = torch.softmax(output[0, -1], dim=0)
        
        idx_next = torch.multinomial(probs, 1).item()
        char = itos[idx_next]

        result += char
        idx = torch.tensor([[idx_next]])

    return result

for i in range(10):
    print(generate(model)+"\n")

CC(Cc2CCC

Ccccc1C(=

CSSSSc1Cl

C(C(CCCC1

CCCCSCC(=

CCCCCCCCO

CC(=OCl)(

CCC(=CCC(

CC1C)N(=O

CCCOcc(CO



C=O)OCC=O

C=OCCO)OC

CC1CCCCCC

CCCCCO)O)

CC=OCCCCC

CCCC=OCCC

C1CCC=OCC

C=OC=OCCO

CCCCCCC=O

C1C1CCC=O

C=O)OCCC=

C=OCCCCCC

CCCCCC=O)

CCCCC=OCC

CC=OCCCCC

COCO)O)OC

CCC=O)OC=

C=OCCC=OC

COC=OCCCC

C=O)OCCCC

C=OCOCCC=

C1C=OCC=O

COCCCCCOC

CCCC1CCOC

COCC=OCCC

C=O)OCCOC

CCCCCCCCO

CCOCCCCCC

C1CCOCCOC

COCC1CC=O